# 03. Entrenamiento de un modelo predictivo

Este cuaderno entrena un clasificador para predecir si un viaje terminará retrasado, usando variables de contexto, clima, eventos y programación.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
import joblib

plt.style.use('seaborn-v0_8-darkgrid')
df = pd.read_csv(Path('data/public_transport_delays.csv'))

df['event_type'] = df['event_type'].fillna('Sin evento')
df['event_presence'] = (df['event_type'] != 'Sin evento').astype(int)
df['date'] = pd.to_datetime(df['date'])
df['time'] = pd.to_datetime(df['time'], format='%H:%M:%S')
df['hour'] = df['time'].dt.hour
df['day_of_week'] = df['date'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

df['scheduled_departure_dt'] = pd.to_datetime(df['date'].astype(str) + ' ' + df['scheduled_departure'])
df['scheduled_arrival_dt'] = pd.to_datetime(df['date'].astype(str) + ' ' + df['scheduled_arrival'])
df['scheduled_trip_duration_min'] = (df['scheduled_arrival_dt'] - df['scheduled_departure_dt']).dt.total_seconds() / 60
df['delay_gap_min'] = df['actual_arrival_delay_min'] - df['actual_departure_delay_min']

feature_columns = [
    'transport_type', 'weather_condition', 'event_type', 'season', 'holiday', 'peak_hour', 'is_weekend',
    'hour', 'day_of_week', 'traffic_congestion_index', 'temperature_C', 'humidity_percent',
    'wind_speed_kmh', 'precipitation_mm', 'event_attendance_est', 'scheduled_trip_duration_min',
    'actual_departure_delay_min', 'actual_arrival_delay_min', 'delay_gap_min'
]

X = df[feature_columns]
y = df['delayed']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

num_features = [
    'traffic_congestion_index', 'temperature_C', 'humidity_percent', 'wind_speed_kmh',
    'precipitation_mm', 'event_attendance_est', 'scheduled_trip_duration_min',
    'actual_departure_delay_min', 'actual_arrival_delay_min', 'delay_gap_min', 'hour', 'day_of_week'
]
cat_features = [
    'transport_type', 'weather_condition', 'event_type', 'season', 'holiday', 'peak_hour', 'is_weekend'
]

preprocess = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), num_features),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_features),
    ]
)

model = Pipeline([
    ('preprocess', preprocess),
    ('classifier', RandomForestClassifier(n_estimators=250, max_depth=12, random_state=42))
])

model.fit(X_train, y_train)
pred = model.predict(X_test)

metrics = {
    'accuracy': round(accuracy_score(y_test, pred), 4),
    'precision': round(precision_score(y_test, pred), 4),
    'recall': round(recall_score(y_test, pred), 4),
    'f1': round(f1_score(y_test, pred), 4),
    'roc_auc': round(roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]), 4),
}
metrics

## Evaluación del modelo

Se revisan métricas de desempeño y la matriz de confusión para interpretar el rendimiento del clasificador.

In [ ]:
cm = confusion_matrix(y_test, pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No retraso', 'Retraso'], yticklabels=['No retraso', 'Retraso'], ax=ax)
ax.set_title('Matriz de confusión')
ax.set_xlabel('Predicción')
ax.set_ylabel('Real')
plt.tight_layout()
plt.show()

metrics

## Guardado del modelo

El pipeline entrenado se serializa para reutilizarlo en nuevas predicciones.

In [ ]:
model_path = Path('models/public_transport_delay_model.joblib')
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(model, model_path)
with open(Path('models/model_metrics.json'), 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

print('Modelo guardado en:', model_path)
print('Métricas guardadas en:', Path('models/model_metrics.json'))

### Resumen del notebook 3

El entrenamiento produjo un clasificador capaz de predecir si un viaje terminará retrasado con métricas útiles para evaluación. El modelo entrenado quedó guardado para uso posterior.